# **Segmentación automática del pavimento en vías usando LangSAM y prompts de texto**

Este script utiliza el modelo **LangSAM** del paquete `samgeo` para realizar la **detección automática de pavimento** en imágenes satelitales `.tif` mediante un **prompt de texto**. Los resultados se exportan tanto en formato raster (máscara `.tif`) como en formato vectorial (`.shp`).

> 🛰️ **Importante**: Este algoritmo se aplica sobre imágenes satelitales que **ya han sido preprocesadas** en pasos anteriores, donde se recortó **únicamente la porción de la vía**, eliminando elementos como vegetación, edificaciones o etiquetas del mapa base. Esto permite que la segmentación se enfoque exclusivamente en el pavimento.

### 🎯 Objetivo del proceso:
Detectar automáticamente **el pavimento** sobre infraestructura vial visible en imágenes satelitales tipo *Terrain*, utilizando descripciones en lenguaje natural como prompt, y generar datos segmentados y vectorizados para análisis posteriores o entrenamiento de modelos.

### ✅ Funcionalidades principales:

- 🧠 **Uso de LangSAM**: modelo de segmentación que combina SAM (Segment Anything Model) con comprensión de lenguaje natural.
- 💬 **Prompt personalizado**: se utiliza la descripción `"pavement"` para detectar vehículos sobre la vía.
- ⚙️ **Ajuste de sensibilidad**:
  - `box_threshold (bt)`: umbral para aceptar predicciones de cajas.
  - `text_threshold (tt)`: umbral de confianza para la relación entre texto y objeto detectado.
- 🔁 **Procesamiento en lote**: recorre una lista de imágenes recortadas y aplica el mismo flujo a cada una.
- 💾 **Exportación de resultados**:
  - Máscara binaria `.tif` con los objetos detectados.
  - Capa vectorial `.shp` convertida desde la máscara.
- 🚫 **Evita reprocesar archivos existentes**: si ya fue segmentada, se salta automáticamente.

### 🛠️ Variables clave:

- `texts`: lista de prompts de texto (por ahora: `"pavement"`).
- `bts` / `tts`: umbrales para segmentación (`box_threshold`, `text_threshold`).
- `sat_list`: lista de imágenes `.tif` que contienen únicamente la vía.
- `sam_fol`: carpeta base de salida organizada por prompt y troncal.
- `troncal`: identificador de la vía o conjunto de imágenes asociadas.

### 🧪 Aplicaciones típicas:

- Detección del pavimento en corredores viales a partir de imágenes públicas.
- Generación de datasets segmentados automáticamente para modelos de visión computacional.
- Análisis del uso del espacio vial o presencia de tráfico sobre rutas específicas.

Este flujo representa un paso avanzado en la construcción de datasets inteligentes, aprovechando modelos multimodales (texto + visión) para crear información geoespacial precisa sin intervención manual.


In [ ]:
import os,glob

#Falta 4
troncal='Troncal9'
BB='22'
z='21'

#Ruta con imágenes satelitales con solo la información de las vías
fol_sat_road='./data/IMAGES/ONLY_ROADS_FILL_Terrz19_Z{}_BB{}/{}'.format(z,BB,troncal)


image_kind=['ONLY_ROADS_FILL','Satellite']

sat_list=glob.glob(os.path.join(fol_sat_road,'*.tif'))
print(len(sat_list))
sam_fol='./data/IMAGES/SAMprompts_z{}_ExtBB{}'.format(z,BB)
os.makedirs(sam_fol, exist_ok=True)

checkpoint_path='./data/sam_vit_h_4b8939.pth'

# Si el checkpoint no existe, lo descarga
url = "https://huggingface.co/spaces/abhishek/StableSAM/resolve/main/sam_vit_h_4b8939.pth"
if not os.path.exists(checkpoint_path):
    print("Descargando checkpoint…")
    r = requests.get(url, stream=True)
    with open(checkpoint_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Descarga completa.")
else:
    print("Checkpoint ya existente, nada que hacer.")

In [ ]:
from samgeo import tms_to_geotiff
from samgeo.text_sam import LangSAM

sam = LangSAM(model_type="sam2-hiera-large")


In [ ]:
from tqdm import tqdm

texts=['pavement']

bts=tts=[0.37]

for text_prompt in texts:
    for bt,tt in zip(bts,tts):
        sam_fol_prompt=os.path.join(sam_fol,text_prompt,troncal,'bt{}_tt{}'.format(bt,tt))
        os.makedirs(sam_fol_prompt, exist_ok=True)
        for i,tif in tqdm(enumerate(sat_list)):
            img_name=os.path.splitext(os.path.basename(tif))[0]
            out=os.path.join(sam_fol_prompt,img_name+'.tif') 
            
            
            if os.path.exists(out):
                continue
            
            try:
                sam.predict(tif, text_prompt, 
                            box_threshold=bt, text_threshold=tt,output=out)
                #shp
                sam.raster_to_vector(out,out.replace('.tif','.shp'))
            except:
                pass
